## Drive mount and unzip

In [2]:
import os
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

if not os.path.isdir('/content/doppler_traces'):
    !unzip "/content/drive/MyDrive/DATASET_SHARP/doppler_traces.zip"

Mounted at /content/drive
Archive:  /content/drive/MyDrive/DATASET_SHARP/doppler_traces.zip
   creating: doppler_traces/
  inflating: doppler_traces/.placeholder  
   creating: doppler_traces/S1a/
  inflating: doppler_traces/S1a/S1a_C_stream_3.txt  
  inflating: doppler_traces/S1a/S1a_H_stream_2.txt  
  inflating: doppler_traces/S1a/S1a_W_stream_1.txt  
  inflating: doppler_traces/S1a/S1a_E_stream_3.txt  
  inflating: doppler_traces/S1a/S1a_R_stream_2.txt  
  inflating: doppler_traces/S1a/S1a_J2_stream_0.txt  
  inflating: doppler_traces/S1a/S1a_J1_stream_3.txt  
  inflating: doppler_traces/S1a/S1a_S_stream_3.txt  
  inflating: doppler_traces/S1a/S1a_J1_stream_0.txt  
  inflating: doppler_traces/S1a/S1a_S_stream_2.txt  
  inflating: doppler_traces/S1a/S1a_C_stream_1.txt  
  inflating: doppler_traces/S1a/S1a_J2_stream_1.txt  
  inflating: doppler_traces/S1a/S1a_J2_stream_3.txt  
  inflating: doppler_traces/S1a/S1a_J1_stream_2.txt  
  inflating: doppler_traces/S1a/S1a_E_stream_1.txt  
  

## Dataset Utility

In [ ]:
import glob
import os
import numpy as np
import pickle
import math as mt
import shutil

def convert_to_number(lab, csi_label_dict):
    lab_num = np.argwhere(np.asarray(csi_label_dict) == lab)[0][0]
    return lab_num

# Works on a single antenna
def create_windows(csi_list, labels_list, sample_length, stride_length):
    csi_matrix_stride = []
    labels_stride = []
    for i in range(len(labels_list)):
        csi_i = csi_list[i]
        label_i = labels_list[i]
        len_csi = csi_i.shape[1]
        for ii in range(0, len_csi - sample_length, stride_length):
            csi_matrix_stride.append(csi_i[:, ii:ii+sample_length])
            labels_stride.append(label_i)
    return csi_matrix_stride, labels_stride

# Works on all antennas together
def create_windows_antennas(csi_list, labels_list, sample_length, stride_length, remove_mean=False):
    # -------------------------------------------------------------------------------------
    #  Split each activity into sliding WINDOWS (keeping all antennas together).
    #  INPUT:
    #    csi_list     : list of N activities; element i = array (n_tot, doppler_freq, time)
    #                   -> ONE entry per activity (already reduced to the train/val/test slice)
    #    labels_list  : list of N labels; one per activity (NOT one per window)
    #    sample_length: window width in Doppler columns (e.g. 340)
    #    stride_length: step between two consecutive windows (e.g. 30)
    #    remove_mean  : if True, center each window by removing its time-mean
    #  OUTPUT:
    #    csi_matrix_stride : list of windows (the samples for the network), many per activity
    #    labels_stride     : label of EACH window (the source activity, replicated)
    #  NB: csi_list and labels_list have the SAME length N and are paired by index.
    # -------------------------------------------------------------------------------------
    csi_matrix_stride = []
    labels_stride = []
    for i in range(len(labels_list)):        # iterate over the N ACTIVITIES
        csi_i = csi_list[i]                  # Doppler spectrum (all antennas) of activity i
        label_i = labels_list[i]             # its label (a single value)
        len_csi = csi_i.shape[2]             # time length = number of available Doppler columns
        # Sliding window: starts at 0, advances by stride_length, stops when
        # there is no more room for a full window of sample_length columns.
        for ii in range(0, len_csi - sample_length, stride_length):
            csi_wind = csi_i[:, :, ii:ii + sample_length, ...]   # crop [ii, ii+sample_length)
            if remove_mean:
                # Optional normalization: mean over time (axis 2) and subtraction
                # -> removes the window's constant offset (here it is False by default).
                csi_mean = np.mean(csi_wind, axis=2, keepdims=True)
                csi_wind = csi_wind - csi_mean
            csi_matrix_stride.append(csi_wind)   # +1 sample (one window)
            labels_stride.append(label_i)        # +1 label = the one of activity i
    return csi_matrix_stride, labels_stride

# Data expansion for num_antennas times
def expand_antennas(file_names, labels, num_antennas):
    file_names_expanded = [item for item in file_names for _ in range(num_antennas)]
    labels_expanded = [item for item in labels for _ in range(num_antennas)]
    stream_ant = np.tile(np.arange(num_antennas), len(labels))
    return file_names_expanded, labels_expanded, stream_ant

''' 

------------------------------------ Convertire in pyTorch ------------------------------------

def create_dataset_single(csi_matrix_files, labels_stride, stream_ant, input_shape, batch_size, shuffle, cache_file,
                          prefetch=True, repeat=True):
    stream_ant = list(stream_ant)
    dataset_csi = tf.data.Dataset.from_tensor_slices((csi_matrix_files, labels_stride, stream_ant))
    py_funct = lambda csi_file, label, stream: (tf.ensure_shape(tf.numpy_function(load_data_single,
                                                                                  [csi_file, stream],
                                                                                  tf.float32), input_shape), label)
    dataset_csi = dataset_csi.map(py_funct)
    dataset_csi = dataset_csi.cache(cache_file)
    if shuffle:
        dataset_csi = dataset_csi.shuffle(len(labels_stride))
    if repeat:
        dataset_csi = dataset_csi.repeat()
    dataset_csi = dataset_csi.batch(batch_size=batch_size)
    if prefetch:
        dataset_csi = dataset_csi.prefetch(buffer_size=1)
    return dataset_csi
'''

## Generate Training Dataset (S1)

In [9]:
import argparse
import glob
import os
import numpy as np
import pickle
import math as mt
import shutil

# --- Parameters (assigned directly to the variables used in the code) -------------
data_dir = '/content/doppler_traces/'   # data root folder (dataset unzipped on Colab)
list_subdir = 'S1a,S1b,S1c'             # experiment sub-directories (comma-separated)
num_packets = 31                        # number of Wi-Fi packets used for one STFT (e.g. 51)
sliding = 1                             # step (in packets) between two consecutive STFTs
window_length = 340                     # number of Doppler columns per window (network input)
stride_length = 30                      # step between two consecutive windows
labels_activities = 'E,J,L,R,W'         # e.g. "E,J,L,R,W" (activity letters)
n_tot = 4                               # number of spatial streams * antennas (channels per activity)

# --- Build the label dictionary ---------------------------------------------------
# From string "E,J,L,R,W" to list ['E','J','L','R','W'].
# The index in the list becomes the numeric label (E->0, J->1, ...).
csi_label_dict = []
for lab_act in labels_activities.split(','):
    csi_label_dict.append(lab_act)

# Activity string used only to build the names of the output folders/files.
activities = np.asarray(labels_activities)

middle = int(np.floor(num_packets / 2))  # central index (not used further, legacy)

# ==================================================================================
#  MAIN LOOP: process one sub-directory (one experiment) at a time
# ==================================================================================
for subdir in list_subdir.split(','):
    exp_dir = data_dir + subdir + '/'  # full path of the experiment folder

    # --- Prepare output folders: train / val / test ------------------------------
    # If they exist, EMPTY them (remove the old files), otherwise CREATE them.
    path_train = exp_dir + 'train_antennas_' + str(activities)
    path_val = exp_dir + 'val_antennas_' + str(activities)
    path_test = exp_dir + 'test_antennas_' + str(activities)
    paths = [path_train, path_val, path_test]
    for pat in paths:
        if os.path.exists(pat):
            remove_files = glob.glob(pat + '/*')
            for f in remove_files:
                os.remove(f)
        else:
            os.mkdir(pat)

    # Any "complete" folder from previous runs: fully deleted.
    path_complete = exp_dir + 'complete_antennas_' + str(activities)
    if os.path.exists(path_complete):
        shutil.rmtree(path_complete)

    # --- Collect the input files --------------------------------------------------
    # Take all files starting with 'S' (SHARP naming), strip the extension
    # ('.txt', last 4 characters) and SORT the names. Sorting is crucial:
    # it guarantees that the n_tot antennas of the same activity are consecutive.
    names = []
    all_files = os.listdir(exp_dir)
    for i in range(len(all_files)):
        if all_files[i].startswith('S'):
            names.append(all_files[i][:-4])
    names.sort()

    # ------------------------------------------------------------------------------
    #  GROUPING BY ACTIVITY
    #  Each activity is stored as n_tot consecutive files (one per antenna/stream).
    #  Here we accumulate the n_tot antennas in 'csi_matrix' and, once complete,
    #  we pack it into 'csi_matrices' together with its label.
    # ------------------------------------------------------------------------------
    csi_matrices = []   # list of arrays (n_tot, doppler_freq, time) -> one per activity
    labels = []         # numeric label of each activity
    lengths = []        # time length (number of Doppler columns) of each activity
    label = 'null'
    prev_label = label
    csi_matrix = []     # buffer: the antennas of the current activity
    processed = False   # True if the current group contains a valid activity
    for i_name, name in enumerate(names):
        # At every multiple of n_tot (except the first) a group of antennas is complete:
        # we close it and store the activity just accumulated.
        if i_name % n_tot == 0 and i_name != 0 and processed:
            ll = csi_matrix[0].shape[1]  # time length of antenna 0

            # Consistency check: all antennas must have the same duration.
            for i_ant in range(1, n_tot):
                if ll != csi_matrix[i_ant].shape[1]:
                    break
            lengths.append(ll)
            csi_matrices.append(np.asarray(csi_matrix))  # stack -> (n_tot, freq, time)
            labels.append(label)
            csi_matrix = []  # clear the buffer for the next activity

        # The activity label is the 5th character of the file name (index 4).
        label = name[4]

        # If the activity is not among the requested ones, skip this file.
        if label not in csi_label_dict:
            processed = False
            continue
        processed = True

        print(name)

        # Convert the activity letter into its numeric index.
        label = convert_to_number(label, csi_label_dict)
        if i_name % n_tot == 0:
            prev_label = label  # start of a new group: fix the expected label
        elif label != prev_label:
            # All antennas of a group must have the SAME label:
            # if not, the ordering/dataset is corrupted.
            print('error in ' + str(name))
            break

        # --- Load the Doppler spectrum of the single antenna ----------------------
        name_file = exp_dir + name + '.txt'
        with open(name_file, "rb") as fp:  # deserialize the pickle
            stft_sum_1 = pickle.load(fp)

        # Remove the mean along axis 0 (for each time column):
        # centers the Doppler spectrum and removes constant/background components.
        stft_sum_1_mean = stft_sum_1 - np.mean(stft_sum_1, axis=0, keepdims=True)

        # Transpose -> layout (doppler_freq, time). Append the antenna to the group.
        csi_matrix.append(stft_sum_1_mean.T)

    # --- Close the LAST group (outside the loop) ----------------------------------
    # The last activity is not closed by the check inside the for, handle it here.
    error = False
    if processed:
        # The final group must contain all n_tot antennas.
        if len(csi_matrix) < n_tot:
            print('error in ' + str(name))
        ll = csi_matrix[0].shape[1]

        # Same duration-consistency check; here it does flag a real error.
        for i_ant in range(1, n_tot):
            if ll != csi_matrix[i_ant].shape[1]:
                print('error in ' + str(name))
                error = True
        if not error:
            lengths.append(ll)
            csi_matrices.append(np.asarray(csi_matrix))
            labels.append(label)

    # ==============================================================================
    #  TEMPORAL SPLIT train / val / test  (60% / 20% / 20%)
    #  The split is done IN TIME on each activity, without mixing activities:
    #  this way windows of the same activity do not straddle two sets.
    #  A "gap" is left between sets to avoid overlap between windows of
    #  different sets (due to the STFT width).
    # ==============================================================================
    if not error:
        lengths = np.asarray(lengths)
        length_min = np.min(lengths)  # minimum duration (informational)

        csi_train = []
        csi_val = []
        csi_test = []
        length_train = []
        length_val = []
        length_test = []
        for i in range(len(labels)):
            ll = lengths[i]  # time duration of activity i

            # --- TRAIN: first 60% of the time sequence ----------------------------
            train_len = int(np.floor(ll * 0.6))
            length_train.append(train_len)
            csi_train.append(csi_matrices[i][:, :, :train_len])

            # --- VAL: next 20%, with gap ceil(num_packets/sliding) ----------------
            # The gap discards the columns "contaminated" by packets shared with train.
            start_val = train_len + mt.ceil(num_packets/sliding)
            val_len = int(np.floor(ll * 0.2))
            length_val.append(val_len)
            csi_val.append(csi_matrices[i][:, :, start_val:start_val + val_len])

            # --- TEST: remainder of the sequence, after another gap ---------------
            start_test = start_val + val_len + mt.ceil(num_packets/sliding)
            length_test.append(ll - val_len - train_len - 2*mt.ceil(num_packets/sliding))
            csi_test.append(csi_matrices[i][:, :, start_test:])

        list_sets_name = ['train', 'val', 'test']
        list_sets = [csi_train, csi_val, csi_test]
        list_sets_lengths = [length_train, length_val, length_test]

        # ==========================================================================
        #  WINDOWING AND SAVING (for each of the 3 sets)
        #  create_windows_antennas splits each sequence into sliding windows of
        #  'window_length' columns, with step 'stride_length'. Each window is one
        #  sample (with all antennas) ready for the neural network.
        # ==========================================================================
        for set_idx in range(3):
            csi_matrices_set, labels_set = create_windows_antennas(list_sets[set_idx], labels, window_length,
                                                                    stride_length, remove_mean=False)

            # Expected number of windows per activity: floor((L - W)/S + 1).
            # Consistency check: must match the number of windows actually generated.
            num_windows = np.floor((np.asarray(list_sets_lengths[set_idx]) - window_length) / stride_length + 1)
            if not len(csi_matrices_set) == np.sum(num_windows):
                print('ERROR - shapes mismatch')

            # --- Save each window as a numbered pickle file -----------------------
            names_set = []
            suffix = '.txt'
            for ii in range(len(csi_matrices_set)):
                name_file = exp_dir + list_sets_name[set_idx] + '_antennas_' + str(activities) + '/' + \
                            str(ii) + suffix
                names_set.append(name_file)
                with open(name_file, "wb") as fp:  # serialize the single window
                    pickle.dump(csi_matrices_set[ii], fp)

            # --- Set summary files ------------------------------------------------
            # labels_*: label of each window
            name_labels = exp_dir + '/labels_' + list_sets_name[set_idx] + '_antennas_' + str(activities) + suffix
            with open(name_labels, "wb") as fp:
                pickle.dump(labels_set, fp)
            # files_*: list of paths of the saved windows (used by the loader)
            name_f = exp_dir + '/files_' + list_sets_name[set_idx] + '_antennas_' + str(activities) + suffix
            with open(name_f, "wb") as fp:
                pickle.dump(names_set, fp)
            # num_windows_*: number of windows per activity (to rebuild the groups)
            name_f = exp_dir + '/num_windows_' + list_sets_name[set_idx] + '_antennas_' + str(activities) + suffix
            with open(name_f, "wb") as fp:
                pickle.dump(num_windows, fp)


S1a_E_stream_0
S1a_E_stream_1
S1a_E_stream_2
S1a_E_stream_3
S1a_J1_stream_0
S1a_J1_stream_1
S1a_J1_stream_2
S1a_J1_stream_3
S1a_J2_stream_0
S1a_J2_stream_1
S1a_J2_stream_2
S1a_J2_stream_3
S1a_L_stream_0
S1a_L_stream_1
S1a_L_stream_2
S1a_L_stream_3
S1a_R_stream_0
S1a_R_stream_1
S1a_R_stream_2
S1a_R_stream_3
S1a_W_stream_0
S1a_W_stream_1
S1a_W_stream_2
S1a_W_stream_3
S1b_E_stream_0
S1b_E_stream_1
S1b_E_stream_2
S1b_E_stream_3
S1b_J1_stream_0
S1b_J1_stream_1
S1b_J1_stream_2
S1b_J1_stream_3
S1b_J2_stream_0
S1b_J2_stream_1
S1b_J2_stream_2
S1b_J2_stream_3
S1b_L_stream_0
S1b_L_stream_1
S1b_L_stream_2
S1b_L_stream_3
S1b_R_stream_0
S1b_R_stream_1
S1b_R_stream_2
S1b_R_stream_3
S1b_W_stream_0
S1b_W_stream_1
S1b_W_stream_2
S1b_W_stream_3
ERROR - shapes mismatch
ERROR - shapes mismatch
ERROR - shapes mismatch
S1c_E_stream_0
S1c_E_stream_1
S1c_E_stream_2
S1c_E_stream_3
S1c_J1_stream_0
S1c_J1_stream_1
S1c_J1_stream_2
S1c_J1_stream_3
S1c_J2_stream_0
S1c_J2_stream_1
S1c_J2_stream_2
S1c_J2_stream_3
S1c_

## Network Utility

In [ ]:
'''

----------------------------------- Convertire in pyTorch -----------------------------------

import tensorflow as tf


def conv2d_bn(x_in, filters, kernel_size, strides=(1, 1), padding='same', activation='relu', bn=False, name=None):
    x = tf.keras.layers.Conv2D(filters, kernel_size, strides=strides, padding=padding, name=name)(x_in)
    if bn:
        bn_name = None if name is None else name + '_bn'
        x = tf.keras.layers.BatchNormalization(axis=3, name=bn_name)(x)
    if activation is not None:
        x = tf.keras.layers.Activation(activation)(x)
    return x


def reduction_a_block_small(x_in, base_name):
    x1 = tf.keras.layers.MaxPool2D((2, 2), strides=(2, 2), padding='valid')(x_in)

    x2 = conv2d_bn(x_in, 5, (2, 2), strides=(2, 2), padding='valid', name=base_name + 'conv2_1_res_a')

    x3 = conv2d_bn(x_in, 3, (1, 1), name=base_name + 'conv3_1_res_a')
    x3 = conv2d_bn(x3, 6, (2, 2), name=base_name + 'conv3_2_res_a')
    x3 = conv2d_bn(x3, 9, (4, 4), strides=(2, 2), padding='same', name=base_name + 'conv3_3_res_a')

    x4 = tf.keras.layers.Concatenate()([x1, x2, x3])
    return x4


def csi_network_inc_res(input_sh, output_sh):
    x_input = tf.keras.Input(input_sh)

    x2 = reduction_a_block_small(x_input, base_name='1st')

    x3 = conv2d_bn(x2, 3, (1, 1), name='conv4')

    x = tf.keras.layers.Flatten()(x3)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Dense(output_sh, activation=None, name='dense2')(x)
    model = tf.keras.Model(inputs=x_input, outputs=x, name='csi_model')
    return model
    '''